# Phase 5 — Multilingual hybrid retrieval

This notebook builds the signed BM25 + multilingual E5 + FAISS retrieval
baseline, applies mandatory authorization and metadata filters, evaluates 50
bootstrap probes, and creates the expert gold-set workbook.

Run **Runtime → Change runtime type → T4 GPU** when available, then select
**Runtime → Run all**. Do not unzip the package manually.

The first run downloads a pinned model of approximately 1.1 GB. It normally
takes 10–25 minutes depending on networking and hardware. Embedding rows are
checkpointed. No Devoteam content is sent to an embedding API or LLM.


## 1. Mount the known clean project


In [ ]:
import hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path

EXPECTED_PACKAGE_SHA256 = "19cc825bba506b79fc1e8dfa2dca4da89d04260a0b79447d5421b179d37979b2"
PACKAGE_FILENAME = "PHASE_5_HYBRID_RETRIEVAL_PACKAGE.zip"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Devoteam internship/Devoteam_AI_CLEAN_PIPELINE")
assert (PROJECT_ROOT / "config" / "project.yaml").exists(), f"Clean project not found: {PROJECT_ROOT}"
PHASE4_ROOT = PROJECT_ROOT / "data" / "canonical" / "20260714T154731Z_129ff982c8" / "phase4_corpus_v1"
assert (PHASE4_ROOT / "_SUCCESS.json").exists(), "Signed Phase 4 result is missing"
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(f"Project root: {PROJECT_ROOT}")


## 2. Verify and install the signed additive package


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, "Phase 5 package hash mismatch"

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), f"Unsafe archive path: {member.filename}"
    package_manifest = json.loads(archive.read("PHASE_5_PACKAGE_MANIFEST.json"))
    assert package_manifest["pipeline_version"] == "phase5_hybrid_retrieval_v1"
    assert package_manifest["expert_gold_set_required_for_promotion"] is True
    for entry in package_manifest["files"]:
        assert hashlib.sha256(archive.read(entry["path"])).hexdigest() == entry["sha256"]
    for member in archive.infolist():
        if member.is_dir() or member.filename == "PHASE_5_PACKAGE_MANIFEST.json":
            continue
        destination = PROJECT_ROOT / member.filename
        packaged_hash = hashlib.sha256(archive.read(member.filename)).hexdigest()
        if destination.exists():
            assert file_sha256(destination) == packaged_hash, f"Refusing to overwrite changed file: {member.filename}"
        else:
            archive.extract(member, PROJECT_ROOT)

print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 5 additive extension installed safely.")


## 3. Install retrieval dependencies and run the complete project tests


In [ ]:
print("Installing/checking retrieval packages — usually 2–5 minutes...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
        "-r", str(PROJECT_ROOT / "requirements" / "phase5.txt"),
    ],
    check=True,
)
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
print("Running the complete foundation-through-retrieval test suite...")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
assert tests.returncode == 0, "Tests failed; Phase 5 did not start."
print("Complete project test suite passed.")


## 4. Build or resume BM25, local embeddings, FAISS, filters, and bootstrap evaluation


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")
if device == "cpu":
    print("GPU is not enabled. The result remains valid, but embedding will take longer.")
print("The pinned E5 model may download now; only model weights enter this runtime.")
print("Devoteam passages are encoded locally and never sent to an external API.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from devoteam_reference_ai.phase5_retrieval import run_phase5

summary = run_phase5(
    project_root=PROJECT_ROOT,
    config_path=PROJECT_ROOT / "config" / "phase5_retrieval.yaml",
    progress=print,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))


## 5. Verify the signed index and all technical gates


In [ ]:
from devoteam_reference_ai.phase5_retrieval import verify_phase5

run_root = Path(summary["run_root"])
verified = verify_phase5(run_root)
assert verified["status"] == "TECHNICAL_PASS"
assert verified["qa_gate"] == "PASS"
assert verified["chunks_indexed"] == 1185
assert verified["embedding_dimensions"] == 768
assert verified["filter_correctness"] == 1.0
assert verified["citation_coverage"] == 1.0
assert verified["external_embedding_api_calls"] == 0
assert verified["external_llm_calls"] == 0
assert verified["production_promotion_status"] == "BLOCKED_PENDING_EXPERT_GOLD_SET"
print("Signed Phase 5 index verification passed.")


## 6. Publish the technical result and honest evaluation boundary


In [ ]:
metrics = json.loads((run_root / "evaluation" / "bootstrap_metrics.json").read_text(encoding="utf-8"))
print("PHASE 5: TECHNICAL PASS")
print(f"Indexed chunks: {verified['chunks_indexed']}")
print(f"BM25 terms: {verified['bm25_terms']}")
print(f"Embedding model: {verified['embedding_model_id']}")
print(f"Bootstrap queries: {verified['bootstrap_queries']}")
for mode in ("bm25", "dense", "hybrid"):
    values = metrics["retrieval_metrics"][mode]
    print(
        f"{mode.upper()}: Recall@10={values['recall_at_10']:.3f} | "
        f"Precision@5={values['precision_at_5']:.3f} | "
        f"MRR={values['mrr']:.3f} | nDCG@10={values['ndcg_at_10']:.3f}"
    )
print(f"Filter correctness: {metrics['filter_correctness']:.3f}")
print(f"Citation coverage: {metrics['citation_coverage']:.3f}")
print(f"Output: {run_root}")
print("IMPORTANT: Bootstrap metrics test plumbing; they are not expert business-quality claims.")
print("Next gate: two Devoteam experts complete EXPERT_GOLD_SET_TEMPLATE.xlsx before model promotion or reranking decisions.")
print("Send this final block for senior review.")
